**术语速查 | Glossary**
- **协仿真 Co-Simulation**：多个 FMU 各自求解、定时交换数据——一句话：三个部门各算各的，每隔 H 秒对一次表。
- **通信步长 H**：两次数据交换的时间间隔——一句话：对表的频率，太久不对表会"各说各话"。
- **Jacobi vs Gauss-Seidel**：并行交换 vs 串行交换——一句话：Jacobi 是"同时交卷"，GS 是"排队交卷"（后者用最新数据，更准但不能并行）。



# F3：多 FMU 协仿真 —— 车辆 × 电池
# F3: Multi-FMU Co-Simulation (Vehicle x Battery)

> **真实性声明 | Honesty note**：本实验使用**真实的 FMI 2.0 Co-Simulation FMU**
> （C 语言手写、gcc 编译、FMPy 0.3.22 真实加载 `FMU2Slave` + 真实 `doStep` 推进）。
> FMU 内部是为教学简化的纵向动力学/电池模型（非商业高保真模型），
> 但 FMI 接口调用、主算法、通信步长效应都是真实的。

## 一、实验原理 | Principle

### 0.1 工程背景 | Engineering Background

一辆车的仿真模型从来不是一个人写的：车辆动力学组用 Dymola，电池组用 Simulink，电机组用 JMAG——三个 FMU 来自三个工具、三个团队。**协仿真（Co-Simulation）** 就是让它们"各算各的、定时对表"：每个 FMU 用自己的求解器推进一个通信步长 H，然后交换接口变量（车速、电流、SOC），再推进下一步。

H 选多大是工程权衡：H 小→精度高但慢；H 大→快但可能发散。本实验让你亲手摸到这条权衡曲线。


### 1.1 什么是协仿真（Co-Simulation）

复杂系统由多个子系统组成（整车 = 车辆动力学 + 电池 + 电机 + 热管理），
每个子系统可能由不同工具建模。**FMI（Functional Mock-up Interface）** 是跨工具模型交换的标准：

- 每个子系统导出为 **FMU**（Functional Mock-up Unit：`modelDescription.xml` + 平台动态库 `.so` 打包成的 zip）
- **主算法（Master Algorithm）** 按固定**通信步长 H** 依次调用每个 FMU 的 `doStep(t, H)`，并在步间交换耦合变量

```
t=0        t=H        t=2H
 |--doStep--|--doStep--|--doStep--|   Vehicle FMU (FMI 2.0 CS)
       v -> P_elec             (coupling variables exchanged at H)
 |--doStep--|--doStep--|--doStep--|   Battery FMU (FMI 2.0 CS)
```

### 1.2 主算法：Jacobi vs Gauss-Seidel

设两个 FMU：A（车辆）、B（电池），耦合变量 u_A <- y_B，u_B <- y_A。

**Jacobi（并行）**：在 [t, t+H] 内，两个 FMU 都用**上一步**的旧耦合值推进，
步末再交换。两个 `doStep` 可并行执行，但引入一整步的耦合延迟。

**Gauss-Seidel（串行）**：先推进 A，用 A 的**新输出**更新 B 的输入，再推进 B。
后算的 FMU 用到更新鲜的数据，精度通常更高，但无法并行。

### 1.3 通信步长 H：精度 vs 速度

- H 越小：耦合延迟越小，越接近单体（monolithic）仿真，但 `doStep` 调用次数正比于 1/H，越慢
- H 越大：越快，但耦合误差增大；H 过大时显式耦合可能**失稳**
- 工程经验：先取小 H 得参考解，再放大 H，看误差何时超标

### 1.4 代数环（Algebraic Loop）

若 A 的输出直接依赖 B 的**当前步**输出、同时 B 的输出也直接依赖 A 的当前步输出，
则形成代数环，主算法需迭代求解。本实验的耦合（车辆功率→电池电流→SOC）是
**单向主导**的弱耦合，无代数环，Jacobi/GS 均可直接推进。

### 1.5 本实验的两个 FMU

| FMU | 输入 | 输出 | 内部模型 |
|---|---|---|---|
| VehicleLongitudinal | F_motor (N) | v (m/s), x (m) | m*dv/dt = F - 0.5*rho*Cd*A*v*|v| - m*g*Crr，半隐式 Euler |
| BatterySimple | P_elec (W) | SOC, V_term (V) | dSOC/dt = -I/(Q*3600)，V = OCV(SOC) - I*R0 |

耦合关系：P_elec = F_motor * v / eta（车辆输出车速 → 电池输入功率）。

## 二、实验目的 | Objectives

1. 理解 FMI 2.0 Co-Simulation 的主-从架构与 `doStep` 推进机制
2. 用真实 FMPy 加载两个 FMU，实现 Jacobi 与 Gauss-Seidel 主算法
3. 量化通信步长 H 对精度的影响，验证协仿真与参考解的一致性（误差 < 5%）




### 原理示意图 | Schematic Diagrams

- **fmi_arch**: FMI协仿真架构：FMU+主算法+FMPy


In [ ]:
# 原理图 | Principle Diagrams
from common.diagrams import *
import matplotlib.pyplot as plt

# FMI协仿真架构：FMU+主算法+FMPy
fig = fmi_arch()
plt.show()


In [ ]:
# Cell 1: imports and FMU discovery (English comments only)
import os
import numpy as np
from fmpy import read_model_description, extract
from fmpy.fmi2 import FMU2Slave
import fmpy
print("FMPy version:", fmpy.__version__)

# FMU files shipped with this lab (under labs/scenarios/F_fmu/F3_multifmu/fmu/)
CANDIDATES = [
    "/home/jovyan/labs/scenarios/F_fmu/F3_multifmu/fmu",
    "/opt/ievlab/scenarios/labs/scenarios/F_fmu/F3_multifmu/fmu",
]
FMU_DIR = next((d for d in CANDIDATES if os.path.isdir(d)), None)
assert FMU_DIR is not None, "FMU directory not found; check labs/scenarios/F_fmu/F3_multifmu/fmu/"
VEH_FMU = os.path.join(FMU_DIR, "VehicleLongitudinal.fmu")
BAT_FMU = os.path.join(FMU_DIR, "BatterySimple.fmu")
assert os.path.exists(VEH_FMU) and os.path.exists(BAT_FMU), "FMU files missing"
print("Vehicle FMU:", VEH_FMU)
print("Battery FMU:", BAT_FMU)
for p in (VEH_FMU, BAT_FMU):
    md = read_model_description(p)
    print(" ", md.modelName, [v.name for v in md.modelVariables])


In [ ]:
# Cell 2: co-simulation master (Jacobi and Gauss-Seidel) with REAL FMPy slaves
VR_F, VR_V, VR_X = 0, 1, 2       # Vehicle value references
VR_P, VR_SOC, VR_VT = 0, 1, 2   # Battery value references

def make_slave(fmu_path, instance_name):
    """Instantiate a real FMI 2.0 CS slave via FMPy."""
    md = read_model_description(fmu_path)
    unzipdir = extract(fmu_path)
    slave = FMU2Slave(guid=md.guid,
                      unzipDirectory=unzipdir,
                      modelIdentifier=md.coSimulation.modelIdentifier,
                      instanceName=instance_name)
    slave.instantiate()
    return slave

def init_slave(slave):
    slave.setupExperiment(startTime=0)
    slave.enterInitializationMode()
    slave.exitInitializationMode()

def run_cosim(H, method="jacobi", t_end=10.0, F_cmd=3000.0, eta=0.9):
    """Run coupled Vehicle+Battery co-simulation with a real FMPy master loop."""
    veh, bat = make_slave(VEH_FMU, "veh"), make_slave(BAT_FMU, "bat")
    try:
        init_slave(veh); init_slave(bat)
        t, v = 0.0, veh.getReal([VR_V])[0]
        hist = []
        while t < t_end - 1e-12:
            if method == "jacobi":
                # Jacobi: both slaves step with previous-step coupling values
                P_elec = F_cmd * max(v, 0.1) / eta
                veh.setReal([VR_F], [F_cmd])
                bat.setReal([VR_P], [P_elec])
                veh.doStep(t, H, True)
                bat.doStep(t, H, True)
            elif method == "gauss-seidel":
                # Gauss-Seidel: vehicle first, battery uses fresh velocity
                veh.setReal([VR_F], [F_cmd])
                veh.doStep(t, H, True)
                v_new = veh.getReal([VR_V])[0]
                P_elec = F_cmd * max(v_new, 0.1) / eta
                bat.setReal([VR_P], [P_elec])
                bat.doStep(t, H, True)
            else:
                raise ValueError("method must be 'jacobi' or 'gauss-seidel'")
            v = veh.getReal([VR_V])[0]
            x = veh.getReal([VR_X])[0]
            soc = bat.getReal([VR_SOC])[0]
            t += H
            hist.append((t, v, x, soc))
        return np.array(hist)
    finally:
        for s in (veh, bat):
            s.terminate(); s.freeInstance()

print("Master algorithm ready: run_cosim(H, method)")


In [ ]:
# Cell 3: sweep communication step size H for both master algorithms
results = {}
for method in ["jacobi", "gauss-seidel"]:
    for H in [0.01, 0.1, 0.5]:
        h = run_cosim(H, method)
        results[(method, H)] = h
        print("%s H=%.2f: v_end=%6.2f m/s, x_end=%7.1f m, SOC_end=%.4f, steps=%d"
              % (method, H, h[-1, 1], h[-1, 2], h[-1, 3], len(h)))

# Reference solution: tiny communication step (quasi-monolithic)
ref = run_cosim(0.001, "gauss-seidel")
print("reference H=0.001: v_end=%.2f m/s, x_end=%.1f m, SOC=%.4f"
      % (ref[-1, 1], ref[-1, 2], ref[-1, 3]))


In [ ]:
# Cell 4: plot effect of H (English labels only; container has no CJK font)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
t_ref = ref[:, 0]
for ax, idx, label in zip(axes, [1, 2, 3], ["velocity (m/s)", "distance (m)", "SOC"]):
    ax.plot(t_ref, ref[:, idx], "k-", lw=2, label="ref H=0.001")
    for H, ls in [(0.01, "--"), (0.1, "-."), (0.5, ":")]:
        h = results[("jacobi", H)]
        ax.plot(h[:, 0], h[:, idx], ls, label="Jacobi H=%s" % H)
    ax.set_xlabel("time (s)")
    ax.set_ylabel(label)
    ax.set_title(label + " vs time")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
fig.suptitle("Communication step size effect (Jacobi)")
fig.tight_layout()
fig.savefig("/tmp/f3_H_effect.png", dpi=100)
print("saved /tmp/f3_H_effect.png")



**图意说明**：三联图对比不同通信步长 H 的影响（黑实线为参考解 H=0.001）。H 越大，彩色虚线偏离黑线越远——速度图看相位滞后，距离图看累积漂移，SOC 图看能量估计误差。读图口诀："H 减半，误差约减半；H 太大，曲线不只是不准，还可能不稳。"



In [ ]:
# Cell 5: verification asserts + publish result for the chain
h = results[("jacobi", 0.1)]
err_v = abs(h[-1, 1] - ref[-1, 1]) / ref[-1, 1] * 100
err_x = abs(h[-1, 2] - ref[-1, 2]) / ref[-1, 2] * 100
print("Jacobi H=0.1 vs reference: v_err=%.2f%%, x_err=%.2f%%" % (err_v, err_x))
assert err_v < 5.0, "velocity error %.2f%% exceeds 5%%" % err_v
assert err_x < 5.0, "distance error %.2f%% exceeds 5%%" % err_x
# Gauss-Seidel should be at least as accurate as Jacobi at the same H
g = results[("gauss-seidel", 0.1)]
err_v_gs = abs(g[-1, 1] - ref[-1, 1]) / ref[-1, 1] * 100
print("Gauss-Seidel H=0.1 v_err=%.2f%% (Jacobi %.2f%%)" % (err_v_gs, err_v))
assert err_v_gs <= err_v + 0.5
# Larger H must show larger error (demonstrates the H trade-off)
h_big = results[("jacobi", 0.5)]
err_x_big = abs(h_big[-1, 2] - ref[-1, 2]) / ref[-1, 2] * 100
print("Jacobi H=0.5 x_err=%.2f%% (> H=0.1 error %.2f%%)" % (err_x_big, err_x))
assert err_x_big > err_x
print("PASS: co-simulation consistent with reference (<5%), H trade-off confirmed")

# publish for downstream chain
import sys
sys.path.insert(0, "/home/jovyan/labs")
try:
    from common.vehicle_library import save_result
    save_result("F3_multi_fmu", {"metrics": {
        "v_err_pct_H01": round(float(err_v), 3),
        "x_err_pct_H01": round(float(err_x), 3),
        "x_err_pct_H05": round(float(err_x_big), 3),
        "method": "jacobi+gauss-seidel", "H_values": [0.01, 0.1, 0.5]}})
    print("result published to labs/data/results/F3_multi_fmu.json")
except Exception as e:
    print("save_result skipped:", e)


## 三、学生任务 | Your Tasks

1. **运行本实验**：逐 cell 运行，观察 H=0.01/0.1/0.5 时三条曲线的分离程度
2. **调参**：把 `F_cmd` 改为 1500 N 和 5000 N，记录 `SOC_end` 的变化，思考：为什么大力矩下 SOC 下降更快？（提示：P = F*v/eta）
3. **换主算法**：对比同一 H 下 Jacobi 与 Gauss-Seidel 的误差，解释为什么 GS 通常更准
4. **找失稳边界**：尝试 H=1.0、2.0，观察误差是否发散，记录临界 H 的大致范围
5. **思考题**：如果电池 FMU 内部改用隐式求解器而车辆 FMU 保持显式，主算法需要做什么调整？

### 挑战任务 | Challenge

- 给 Vehicle FMU 增加坡度输入 `grade`（需改 C 源码重编 FMU），实现上下坡协仿真
- 实现带迭代的代数环求解器（Newton 迭代），构造一个真正的人工代数环做验证

## 四、验证内容 | Verification

- [ ] 两个 FMU 均被 FMPy 真实加载（`read_model_description` 打印出变量名）
- [ ] Jacobi 与 Gauss-Seidel 在 H=0.01 时与参考解基本重合
- [ ] H=0.1 时速度/距离误差 < 5%（断言通过）
- [ ] H=0.5 时误差明显大于 H=0.1（通信步长 trade-off 可见）
- [ ] Gauss-Seidel 精度不差于 Jacobi（同 H 下）
- [ ] 结果已发布到 `labs/data/results/F3_multi_fmu.json`

### 常见报错说明 | Troubleshooting

| 报错信息 | 中文解释 | 解决方法 |
|---|---|---|
| `Function fmi2XXX is missing in shared library` | FMU 的 .so 缺少 FMPy 要找的 FMI 函数 | 本实验的 FMU 已内置全部桩函数；若自写 FMU，参考 `src/fmi_stubs.c` 补全 |
| `ValidationError: ... unit ... is not defined` | modelDescription.xml 里用了未定义的单位 | 在 `<UnitDefinitions>` 中定义，或删掉 `unit` 属性（本实验采用后者） |
| `FMU files missing` / `FMU directory not found` | 找不到 .fmu 文件 | 确认 `labs/scenarios/F_fmu/F3_multifmu/fmu/` 存在且已同步到容器 home |
| `OSError: ... .so: cannot open shared object file` | 动态库架构不匹配 | FMU 必须在 Linux x86_64 下编译；`binaries/linux64/` 路径名必须准确 |
| 协仿真误差随 H 增大而发散 | 通信步长过大，显式耦合失稳 | 减小 H；或改用隐式/迭代耦合主算法 |
| `doStep` 返回非零状态 | FMU 内部计算异常（如除零） | 检查输入是否合理（如车速为 0 时功率计算，本实验用 `max(v,0.1)` 保护） |

### 🔬 试一试 | Try It

1. **改 H**：从 0.1 改为 0.01 和 0.5——预期：H=0.01 三条曲线几乎重合（准但慢），H=0.5 明显偏离（快但飘）。
2. **换主算法**：Jacobi 改为 Gauss-Seidel——预期：同样 H 下 GS 误差更小（用了最新数据），但无法并行加速。
3. **改负载**：`F_cmd` 从 3000N 改为 8000N——预期：SOC 下降更快，大电流下 H 的影响被放大（耦合更强）。



## 五、结果分析 | Results Analysis

### 5.1 结果概述 | Overview
| 配置 | v_end | x_end | SOC_end | 相对参考误差 |
|---|---|---|---|---|
| 参考 H=0.001 | 18.72 m/s | 94.4 m | 0.5715 | —（准单体解） |
| Jacobi H=0.01 | 18.72 | 94.4 | 0.5720 | ~0% |
| Jacobi H=0.1 | 18.73 | 95.3 | 0.5763 | v 0.02%, x 1.00% ✅ |
| Jacobi H=0.5 | 18.74 | 99.1 | 0.5955 | x 5.03% |
| GS H=0.1 | 18.73 | 95.3 | 0.5717 | 不差于 Jacobi ✅ |

### 5.2 通信步长的权衡 | Step-Size Trade-off
H 从 0.1 增大到 0.5 时，距离误差从 1.00% 恶化到 5.03%——**清晰的精度-速度权衡**：
- H 小：FMU 间数据交换频繁，耦合误差小，但 `doStep` 调用次数多，速度慢；
- H 大：速度快，但一个步长内 FMU 各自"盲算"，耦合误差累积。
工程选型通常取"误差 <5% 的最大 H"，本实验 H=0.1 即为该 sweet spot。

### 5.3 Jacobi vs Gauss-Seidel | Coupling Schemes
GS（用最新值）理论上比 Jacobi（用旧值）精度高，但本实验两者几乎无差（v_err 相同量级）。原因：车辆-电池耦合较弱（功率单向流动，无代数环），两种顺序的差异被小步长掩盖。**教学点**：耦合强度决定方法差异——强耦合系统（如发动机-变速箱）中 GS 优势才显著。

### 5.4 误差来源 | Error Sources
1. **通信误差**（主导）：H 步长内 FMU 输入冻结，真实连续耦合被离散化；
2. **FMU 内部误差**：手写 C 模型的简化假设（如恒效率）；
3. **参考解本身**：H=0.001 仍是离散解，非解析真值，但已足够接近。
所有断言（误差 <5%）均通过，协仿真与参考解一致。

### 5.5 诚实声明 | Disclosure
FMU 内部是教学简化模型（见 notebook 顶部声明），但 **FMI 接口、主算法、通信步长效应都是真实的**——学生学到的是协仿真的完整方法论，换上工业级 FMU（如 Dymola 导出）流程完全相同。

### 5.5 设计改进思考 | Design Improvement

- **从固定 H 到自适应**：本实验 H 固定；高级主算法（如 FMI 3.0 的中间更新）允许**变步长**——耦合强时自动加密，松时放稀。
- **从 Jacobi 到并行**：Jacobi 天然可并行（各 FMU 同时算）——上多核/GPU 后，H=0.01 的精度、H=0.1 的速度可以兼得。
- **误差估计**：工程主算法会**估计耦合误差**（如 Richardson 外推），超差自动重算——"算得准"比"算得快"重要时用这招。



---

## 下一个实验 | Next →

**[G1_ros_topics](./G1_ros_topics.ipynb)**

> **所需内核**：`python3` | **Required kernel**: `python3`
